<a href="https://colab.research.google.com/github/DineshGujjeti/sentiment-analysis-nlp/blob/main/sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Import Libraries

In [ ]:
import pandas as pd
import numpy as np

import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


##Load Dataset

In [ ]:
url = "https://raw.githubusercontent.com/dD2405/Twitter_Sentiment_Analysis/master/train.csv"

data = pd.read_csv(url)

data.head()

,id,label,tweet
0,1,0,@user when a father is dysfunctional and is s...
1,2,0,@user @user thanks for #lyft credit i can't us...
2,3,0,bihday your majesty
3,4,0,#model i love u take with u all the time in ...
4,5,0,factsguide: society now #motivation


##Prepare the Dataset

In [ ]:
data = data[['label','tweet']]
data.head()

#Convert labels:
data['label'] = data['label'].map({0:"Positive", 1:"Negative"})

##Text Cleaning

In [ ]:
stop_words = set(stopwords.words('english'))

def clean_text(text):

    text = text.lower()

    text = re.sub(r'http\S+', '', text)

    text = re.sub(r'[^a-zA-Z]', ' ', text)

    words = text.split()

    words = [word for word in words if word not in stop_words]

    return " ".join(words)

data['clean_text'] = data['tweet'].apply(clean_text)

##Train Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    data['clean_text'],
    data['label'],
    test_size=0.2,
    random_state=42
)

##TF-IDF Vectorization

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000)

X_train_vec = vectorizer.fit_transform(X_train)

X_test_vec = vectorizer.transform(X_test)

##Train Model

In [ ]:
model = LogisticRegression(class_weight='balanced')
model.fit(X_train_vec, y_train)

LogisticRegression(class_weight='balanced')

##Evaluate Model

In [ ]:
y_pred = model.predict(X_test_vec)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

    Negative       0.46      0.81      0.58       456
    Positive       0.98      0.93      0.95      5937

    accuracy                           0.92      6393
   macro avg       0.72      0.87      0.77      6393
weighted avg       0.95      0.92      0.93      6393



##Test Custom Text

In [ ]:
review = ["I love this product"]

review_vec = vectorizer.transform(review)

prediction = model.predict(review_vec)

print(prediction)

['Positive']


In [ ]:
review = ["I am very happy today"]

review_vec = vectorizer.transform(review)

prediction = model.predict(review_vec)

print(prediction)

['Positive']


In [ ]:
review = ["This is terrible and I hate it"]

review_vec = vectorizer.transform(review)

prediction = model.predict(review_vec)

print(prediction)

['Negative']
